# 03 — Gold Analytics Layer
## Well Production Forecasting & Performance Analytics System

**Medallion Architecture — Gold Layer**

Computes analytics-ready KPI tables:
1. `gold_operator_performance` — oil/gas production, flaring intensity, well count per operator
2. `gold_basin_production_trends` — monthly basin production with MoM/YoY growth rates
3. `gold_flaring_intensity` — flaring volume, intensity ratio, categorical ranking
4. `gold_ethane_dry_gas` — liquid-rich vs dry gas ratios by operator/year
5. `gold_well_summary` — per-well cumulative production, peak month, active months

All tables are optimized for BI dashboards and ad-hoc SQL queries.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import pandas as pd
import numpy as np

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .master('local[*]')
    .appName('WellAnalytics-Gold')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.databricks.delta.schema.autoMerge.enabled', 'true')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.driver.memory', '4g')
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel('WARN')

sns.set_theme(style='whitegrid', palette='muted')
PLOT_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data', 'plots')
os.makedirs(PLOT_DIR, exist_ok=True)
print(f'Spark {spark.version} ready | Plots -> {PLOT_DIR}')

## Step 1 — Compute Gold KPI Tables

In [ ]:
from src.gold_analytics import run_gold

counts = run_gold(spark)
print('\nGold table row counts:')
for t, n in counts.items():
    print(f'  {t:<40s} {n:>6,}')

## Step 2 — Visualization: Top 10 Operators by Oil Production

In [ ]:
from src.spark_session import gold_path

op_perf = spark.read.format('delta').load(gold_path('gold_operator_performance')).toPandas()
top10 = op_perf.nlargest(10, 'total_oil_bbl').sort_values('total_oil_bbl')

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top10['operator'], top10['total_oil_bbl'] / 1e6,
               color=sns.color_palette('Blues_r', 10))
ax.set_xlabel('Total Oil Production (Million BBL)', fontsize=12)
ax.set_title('Top 10 Operators — Cumulative Oil Production (2018-2023)', fontsize=14, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}M'))
for bar in bars:
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.1f}M', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, '01_top10_operators_oil.png'), dpi=150)
plt.show()
print('Saved: 01_top10_operators_oil.png')

## Step 3 — Visualization: Basin Production Trends

In [ ]:
basin = spark.read.format('delta').load(gold_path('gold_basin_production_trends')).toPandas()
basin['production_month'] = pd.to_datetime(basin['production_month'])

oil_basin = basin[basin['oil_and_gas_group'] == 'O'].copy()

fig, ax = plt.subplots(figsize=(12, 6))
for play, grp in oil_basin.groupby('shale_play'):
    grp_sorted = grp.sort_values('production_month')
    ax.plot(grp_sorted['production_month'], grp_sorted['total_production'] / 1e3,
            label=play, linewidth=2)

ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Total Oil Production (K BBL)', fontsize=12)
ax.set_title('Basin Oil Production Trends (2018-2023)', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.xaxis.set_major_locator(plt.MaxNLocator(12))
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, '02_basin_production_trends.png'), dpi=150)
plt.show()
print('Saved: 02_basin_production_trends.png')

## Step 4 — Visualization: Flaring Intensity by Operator

In [ ]:
flaring = spark.read.format('delta').load(gold_path('gold_flaring_intensity')).toPandas()
flaring = flaring.dropna(subset=['flaring_intensity_ratio']).sort_values('flaring_intensity_ratio', ascending=False)

color_map = {'Low': '#2ecc71', 'Medium': '#f39c12', 'High': '#e74c3c'}
colors = flaring['flaring_category'].map(color_map).fillna('#95a5a6')

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(flaring['operator'], flaring['flaring_intensity_ratio'], color=colors)
ax.axvline(x=5, color='orange', linestyle='--', linewidth=1.2, label='Medium threshold (5%)')
ax.axvline(x=15, color='red', linestyle='--', linewidth=1.2, label='High threshold (15%)')
ax.set_xlabel('Flaring Intensity (% of Gas Production)', fontsize=12)
ax.set_title('Operator Flaring Intensity — ESG KPI', fontsize=14, fontweight='bold')
ax.legend(fontsize=9)
from matplotlib.patches import Patch
legend_els = [Patch(color=v, label=k) for k, v in color_map.items()]
ax.legend(handles=legend_els, loc='lower right', fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, '03_flaring_intensity.png'), dpi=150)
plt.show()
print('Saved: 03_flaring_intensity.png')

## Step 5 — Visualization: Ethane vs Dry Gas Ratios

In [ ]:
edg = spark.read.format('delta').load(gold_path('gold_ethane_dry_gas')).toPandas()
edg_agg = (
    edg.groupby('operator')[['liquid_rich_production', 'dry_gas_production']]
    .sum()
    .reset_index()
    .assign(total=lambda d: d['liquid_rich_production'] + d['dry_gas_production'])
    .sort_values('total', ascending=False)
    .head(12)
)

fig, ax = plt.subplots(figsize=(11, 6))
x = range(len(edg_agg))
ax.bar(x, edg_agg['liquid_rich_production'] / 1e6, label='Liquid-Rich (Oil)', color='#3498db')
ax.bar(x, edg_agg['dry_gas_production'] / 1e6,
       bottom=edg_agg['liquid_rich_production'] / 1e6,
       label='Dry Gas', color='#e67e22')
ax.set_xticks(list(x))
ax.set_xticklabels(edg_agg['operator'], rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Production (Million units)', fontsize=12)
ax.set_title('Ethane (Liquid-Rich) vs Dry Gas Production by Operator', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, '04_ethane_vs_dry_gas.png'), dpi=150)
plt.show()
print('Saved: 04_ethane_vs_dry_gas.png')

## Step 6 — Visualization: Well Count by Basin

In [ ]:
well_summary = spark.read.format('delta').load(gold_path('gold_well_summary')).toPandas()
basin_counts = well_summary.groupby('shale_play')['api_number'].count()

fig, ax = plt.subplots(figsize=(7, 7))
wedges, texts, autotexts = ax.pie(
    basin_counts,
    labels=basin_counts.index,
    autopct='%1.1f%%',
    colors=sns.color_palette('Set2', len(basin_counts)),
    startangle=90,
    pctdistance=0.8,
)
for t in autotexts:
    t.set_fontsize(11)
ax.set_title('Well Count Distribution by Basin', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, '05_well_count_by_basin.png'), dpi=150)
plt.show()
print('Saved: 05_well_count_by_basin.png')

## Step 7 — Ad-hoc SQL Showcase

In [ ]:
from src.spark_session import gold_path

spark.read.format('delta').load(gold_path('gold_operator_performance')).createOrReplaceTempView('gold_operator_performance')
spark.read.format('delta').load(gold_path('gold_basin_production_trends')).createOrReplaceTempView('gold_basin_production_trends')
spark.read.format('delta').load(gold_path('gold_flaring_intensity')).createOrReplaceTempView('gold_flaring_intensity')

print('=== Top 5 Operators by Flaring Intensity ===')
spark.sql("""
    SELECT operator, ROUND(flaring_intensity_ratio, 2) AS flaring_pct,
           flaring_category, flaring_rank
    FROM gold_flaring_intensity
    WHERE flaring_intensity_ratio IS NOT NULL
    ORDER BY flaring_rank
    LIMIT 5
""").show()

print('=== YoY Basin Production Growth (latest year) ===')
spark.sql("""
    SELECT shale_play, oil_and_gas_group,
           ROUND(AVG(yoy_growth_pct), 2) AS avg_yoy_growth_pct
    FROM gold_basin_production_trends
    WHERE yoy_growth_pct IS NOT NULL
    GROUP BY shale_play, oil_and_gas_group
    ORDER BY shale_play, oil_and_gas_group
""").show()